In [ ]:
"""
Confusion Matrix Plotter for Fruit Classification Errors
=========================================================
Usage:
    python plot_confusion_matrix.py --csv path/to/results.csv --out confusion_matrix.png

CSV columns used:
    gt_fruit               - ground truth fruit label
    extracted_fruit        - what the model said
    fruit_answer_profiling - CORRECT | LEAK | DENIAL | RESCUE | OTHER

Mapping to confusion matrix columns:
    CORRECT  -> skipped entirely (not an error)
    RESCUE   -> pred = gt  (diagonal)
    DENIAL   -> pred = "deny"
    LEAK     -> pred = "leak"
    OTHER    -> pred = normalise(extracted_fruit); unknown vocab -> "other"
"""

import argparse
import re
import csv

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── Canonical categories ─────────────────────────────────────────────────────
FRUITS = [
    "apple", "banana", "coconut", "grape", "grapefruit",
    "mango", "peach", "pear", "pineapple", "strawberry", "watermelon",
]
SPECIAL = ["deny", "leak", "other"]
ALL_LABELS = FRUITS + SPECIAL

DISPLAY_LABELS = [
    "Apple", "Banana", "Coconut", "Grape", "Grapefruit",
    "Mango", "Peach", "Pear", "Pineapple", "Strawberry", "Watermelon",
    "deny", "leak", "other",
]

BOLD_LABELS = {"Apple", "Banana", "deny", "leak"}

# Explicit surface-form variants only — no guessing.
NORMALISE_MAP: dict = {
    "apples":       "apple",
    "bananas":      "banana",
    "coconuts":     "coconut",
    "grapes":       "grape",
    "grapefruits":  "grapefruit",
    "mangoes":      "mango",
    "mangos":       "mango",
    "peaches":      "peach",
    "pears":        "pear",
    "pineapples":   "pineapple",
    "strawberries": "strawberry",
    "watermelons":  "watermelon",
}


def normalise(raw: str) -> str:
    """
    Lowercase + strip non-alpha -> check NORMALISE_MAP -> check ALL_LABELS.
    Anything not recognised -> 'other'. No fuzzy matching.
    """
    s = re.sub(r"[^a-z]", "", raw.strip().lower())
    if s in NORMALISE_MAP:
        return NORMALISE_MAP[s]
    if s in ALL_LABELS:
        return s
    return "other"


def load_matrix(csv_path: str) -> np.ndarray:
    """
    Parse the CSV and return an (n_labels x n_labels) count matrix.
    Rows = true label, Columns = predicted label.
    """
    label_idx = {label: i for i, label in enumerate(ALL_LABELS)}
    matrix = np.zeros((len(ALL_LABELS), len(ALL_LABELS)), dtype=int)

    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            profiling = row.get("fruit_answer_profiling", "").strip().upper()

            # Skip correct answers — this is an error-only matrix
            if profiling == "CORRECT":
                continue

            gt = normalise(row.get("gt_fruit", ""))

            if profiling == "RESCUE":
                pred = gt                              # counts on diagonal
            elif profiling == "DENIAL":
                pred = "deny"
            elif profiling == "LEAK":
                pred = "leak"
            else:                                      # OTHER or unexpected
                pred = normalise(row.get("extracted_fruit", ""))

            r = label_idx.get(gt, label_idx["other"])
            c = label_idx.get(pred, label_idx["other"])
            matrix[r, c] += 1

    return matrix


def plot_matrix(
    matrix: np.ndarray,
    out_path: str,
    title: str = "Fruit Classifier — Error Confusion Matrix",
) -> plt.Figure:
    """Render and save the confusion matrix."""
    n = len(ALL_LABELS)

    # Compact cell size with high dpi = tight grid, large rendered fonts
    cell = 0.32
    fig, ax = plt.subplots(figsize=(n * cell + 3.2, n * cell + 2.0))

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # cmap = mcolors.LinearSegmentedColormap.from_list("fruit_heat", [
    #     "#ffffff", "#fee5d9", "#fcae91", "#fb6a4a", "#cb181d",
    # ])

    cmap = plt.cm.Oranges

    vmax = matrix.max() if matrix.max() > 0 else 1
    im = ax.imshow(matrix.astype(float), cmap=cmap, vmin=0, vmax=vmax,
                   aspect="equal", interpolation="nearest")

    # Cell text — large, bold on diagonal
    for i in range(n):
        for j in range(n):
            val = matrix[i, j]
            txt_color = "white" if (val / vmax) > 0.55 else "#222222"
            ax.text(j, i, str(val), ha="center", va="center",
                    fontsize=13, color=txt_color,
                    fontweight="bold" if i == j else "normal",
                    fontfamily="monospace")

    # Grid lines
    for x in np.arange(-0.5, n, 1):
        ax.axhline(x, color="#cccccc", linewidth=0.6, zorder=3)
        ax.axvline(x, color="#cccccc", linewidth=0.6, zorder=3)

    # Divider between fruit labels and special labels
    divider_kw = dict(color="#555555", linewidth=1.4, linestyle="--", alpha=0.6, zorder=5)
    ax.axhline(len(FRUITS) - 0.5, **divider_kw)
    ax.axvline(len(FRUITS) - 0.5, **divider_kw)

    def fmt(name: str) -> str:
        return f"$\\bf{{{name}}}$" if name in BOLD_LABELS else name

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels([fmt(l) for l in DISPLAY_LABELS], rotation=45, ha="right",
                       fontsize=13, color="#222222")
    ax.set_yticklabels([fmt(l) for l in DISPLAY_LABELS], rotation=0,
                       fontsize=13, color="#222222")
    ax.tick_params(axis="both", which="both", length=0, pad=5)

    ax.set_xlabel("Predicted Label", fontsize=14, color="#222222", labelpad=12)
    ax.set_ylabel("True Label", fontsize=14, color="#222222", labelpad=12)
    ax.set_title(title, fontsize=15, color="#111111", pad=7)

    for spine in ax.spines.values():
        spine.set_edgecolor("#cccccc")

    # cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, shrink=0.8)
    # cbar.ax.tick_params(colors="#333333", labelsize=11)
    # cbar.outline.set_edgecolor("#cccccc")
    # cbar.set_label("Count", color="#333333", fontsize=12)

    plt.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Saved -> {out_path}")
    return fig


# def main():
#     parser = argparse.ArgumentParser(description="Plot fruit error confusion matrix.")
#     parser.add_argument("--csv",   required=True,                    help="Path to results CSV")
#     parser.add_argument("--out",   default="confusion_matrix.png",   help="Output image path")
#     parser.add_argument("--title", default="Fruit Classifier — Error Confusion Matrix")
#     args = parser.parse_args()

#     print(f"Reading {args.csv} ...")
#     matrix = load_matrix(args.csv)
#     plot_matrix(matrix, args.out, title=args.title)


# if __name__ == "__main__":
#     main()

In [ ]:
from matplotlib.pyplot import plot


csv_path = "<REPO_ROOT>/data/automated_gemini_acc_evals/conf_matrix_new_script/lov_05b/confusion_matrix_results.csv"
matrix = load_matrix(csv_path)

plot_matrix(matrix, "<REPO_ROOT>/data/plots/conf_matrix_fc_blocking/lov05b.pdf", title="LOV-0.5B")

In [ ]:
# Fixed

In [ ]:
"""
Confusion Matrix Plotter for Fruit Classification
=========================================================
Usage:
    python plot_confusion_matrix.py --csv path/to/results.csv --out confusion_matrix.png

CSV columns used:
    gt_fruit               - ground truth fruit label
    extracted_fruit        - what the model said
    fruit_answer_profiling - CORRECT | LEAK | DENIAL | RESCUE | OTHER

Mapping to confusion matrix columns:
    CORRECT & RESCUE -> pred = gt  (diagonal)
    DENIAL           -> pred = "deny"
    LEAK             -> pred = "leak"
    OTHER            -> pred = normalise(extracted_fruit); unknown vocab -> "other"
"""

import argparse
import re
import csv

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── Canonical categories ─────────────────────────────────────────────────────
FRUITS = [
    "apple", "banana", "coconut", "grape", "grapefruit",
    "mango", "peach", "pear", "pineapple", "strawberry", "watermelon",
]
SPECIAL = ["deny", "leak", "other"]
ALL_LABELS = FRUITS + SPECIAL

DISPLAY_LABELS = [
    "Apple", "Banana", "Coconut", "Grape", "Grapefruit",
    "Mango", "Peach", "Pear", "Pineapple", "Strawberry", "Watermelon",
    "deny", "leak", "other",
]

BOLD_LABELS = {"Apple", "Banana", "deny", "leak"}

# Explicit surface-form variants only — no guessing.
NORMALISE_MAP: dict = {
    "apples":       "apple",
    "bananas":      "banana",
    "coconuts":     "coconut",
    "grapes":       "grape",
    "grapefruits":  "grapefruit",
    "mangoes":      "mango",
    "mangos":       "mango",
    "peaches":      "peach",
    "pears":        "pear",
    "pineapples":   "pineapple",
    "strawberries": "strawberry",
    "watermelons":  "watermelon",
}


def normalise(raw: str) -> str:
    """
    Lowercase + strip non-alpha -> check NORMALISE_MAP -> check ALL_LABELS.
    Anything not recognised -> 'other'. No fuzzy matching.
    """
    s = re.sub(r"[^a-z]", "", raw.strip().lower())
    if s in NORMALISE_MAP:
        return NORMALISE_MAP[s]
    if s in ALL_LABELS:
        return s
    return "other"


def load_matrix(csv_path: str) -> np.ndarray:
    """
    Parse the CSV and return an (n_labels x n_labels) count matrix.
    Rows = true label, Columns = predicted label.
    """
    label_idx = {label: i for i, label in enumerate(ALL_LABELS)}
    matrix = np.zeros((len(ALL_LABELS), len(ALL_LABELS)), dtype=int)

    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            profiling = row.get("fruit_answer_profiling", "").strip().upper()
            gt = normalise(row.get("gt_fruit", ""))

            # CORRECT and RESCUE both count on the diagonal
            if profiling in ("CORRECT", "RESCUE"):
                pred = gt                              
            elif profiling == "DENIAL":
                pred = "deny"
            elif profiling == "LEAK":
                pred = "leak"
            else:                                      # OTHER or unexpected
                pred = normalise(row.get("extracted_fruit", ""))

            r = label_idx.get(gt, label_idx["other"])
            c = label_idx.get(pred, label_idx["other"])
            matrix[r, c] += 1

    return matrix


def plot_matrix(
    matrix: np.ndarray,
    out_path: str,
    title: str = "Fruit Classifier — Confusion Matrix",
) -> plt.Figure:
    """Render and save the trimmed N-3 x N confusion matrix."""
    n_cols = len(ALL_LABELS)
    n_rows = len(FRUITS)  # We drop the last 3 rows for plotting

    # Slice the matrix to only plot true labels for fruits
    matrix_plot = matrix[:n_rows, :]

    # Compact cell size with high dpi = tight grid, large rendered fonts
    cell = 0.32
    fig, ax = plt.subplots(figsize=(n_cols * cell + 3.2, n_rows * cell + 2.0))

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    cmap = plt.cm.Oranges

    vmax = matrix_plot.max() if matrix_plot.max() > 0 else 1
    im = ax.imshow(matrix_plot.astype(float), cmap=cmap, vmin=0, vmax=vmax,
                   aspect="equal", interpolation="nearest")

    # Cell text — large, bold on diagonal
    for i in range(n_rows):
        for j in range(n_cols):
            val = matrix_plot[i, j]
            txt_color = "white" if (val / vmax) > 0.55 else "#222222"
            ax.text(j, i, str(val), ha="center", va="center",
                    fontsize=13, color=txt_color,
                    fontweight="bold" if i == j else "normal",
                    fontfamily="monospace")

    # Grid lines
    for x in np.arange(-0.5, n_cols, 1):
        ax.axvline(x, color="#cccccc", linewidth=0.6, zorder=3)
    for y in np.arange(-0.5, n_rows, 1):
        ax.axhline(y, color="#cccccc", linewidth=0.6, zorder=3)

    # Divider between fruit labels and special labels (vertical only now)
    divider_kw = dict(color="#555555", linewidth=1.4, linestyle="--", alpha=0.6, zorder=5)
    ax.axvline(len(FRUITS) - 0.5, **divider_kw)

    def fmt(name: str) -> str:
        return f"$\\bf{{{name}}}$" if name in BOLD_LABELS else name

    ax.set_xticks(range(n_cols))
    ax.set_yticks(range(n_rows))
    
    # X labels get all columns, Y labels only get the first n_rows (fruits)
    ax.set_xticklabels([fmt(l) for l in DISPLAY_LABELS], rotation=45, ha="right",
                       fontsize=13, color="#222222")
    ax.set_yticklabels([fmt(l) for l in DISPLAY_LABELS[:n_rows]], rotation=0,
                       fontsize=13, color="#222222")
    ax.tick_params(axis="both", which="both", length=0, pad=5)

    ax.set_xlabel("Predicted Label", fontsize=14, color="#222222", labelpad=12)
    ax.set_ylabel("True Label", fontsize=14, color="#222222", labelpad=12)
    ax.set_title(title, fontsize=15, color="#111111", pad=7)

    for spine in ax.spines.values():
        spine.set_edgecolor("#cccccc")

    plt.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Saved -> {out_path}")
    return fig


# def main():
#     parser = argparse.ArgumentParser(description="Plot fruit error confusion matrix.")
#     parser.add_argument("--csv",   required=True,                    help="Path to results CSV")
#     parser.add_argument("--out",   default="confusion_matrix.png",   help="Output image path")
#     parser.add_argument("--title", default="Fruit Classifier — Confusion Matrix")
#     args = parser.parse_args()

#     print(f"Reading {args.csv} ...")
#     matrix = load_matrix(args.csv)
#     plot_matrix(matrix, args.out, title=args.title)


# if __name__ == "__main__":
#     main()

In [ ]:
from matplotlib.pyplot import plot


csv_path = "<REPO_ROOT>/data/automated_gemini_acc_evals/conf_matrix_new_script/qvl_3b/confusion_matrix_results.csv"
matrix = load_matrix(csv_path)

plot_matrix(matrix, "<REPO_ROOT>/data/plots/conf_matrix_fc_blocking/qvl_3b.pdf", title="QVL-3B")